# EV Charging Station Placement: A Capacitated Facility Location Approach
**Author:** Omer Toledo | **Date:** May 2026

This notebook implements the full pipeline for optimizing public EV charging station placement in Mountain View, California using a Capacitated Facility Location Problem (CFLP) formulation.

**Pipeline order:**
1. Demand points — Census block groups + ACS vehicle ownership + tenure data
2. Home-Charging Vulnerability Index (HCVI)
3. Candidate sites — OSM parking lots + feasibility tiers
4. Distance matrix
5. MILP solver — Scenario A (uniform 10% adoption)
6. MILP solver — Scenario B (tenure-weighted demand)
7. Heuristic — Coverage-first greedy + simulated annealing
8. Existing network baseline + incremental expansion
9. Equity analysis — HCVI-stratified metrics + equity-constrained MILP
10. Sensitivity grid — K, Q, r, adoption rate
11. Stochastic demand analysis — 6 scenarios, regret, CVaR, site stability
12. Road-network distance validation
13. Interactive solution maps


## 1. Demand Points
Downloads Census TIGER/Line block group shapefiles, filters to Mountain View's incorporated boundary, pulls ACS vehicle ownership (B25044) and tenure (B25003) data, and computes two demand scenarios.

In [ ]:
# ================================
# STEP 0: Install dependencies
# ================================
!pip install -q osmnx

# ================================
# STEP 1: Download Census block group shapefile for California
# ================================
!wget -q https://www2.census.gov/geo/tiger/TIGER2023/BG/tl_2023_06_bg.zip
!unzip -q -n tl_2023_06_bg.zip -d bg_data

# ================================
# STEP 2: Load shapefile and filter to Mountain View (city boundary)
# ================================
import geopandas as gpd
import osmnx as ox

bg = gpd.read_file("bg_data/tl_2023_06_bg.shp")
mv_boundary = ox.geocode_to_gdf("Mountain View, California, USA")

bg_3310 = bg.to_crs(epsg=3310)
mv_3310 = mv_boundary.to_crs(epsg=3310)
mv_polygon = mv_3310.geometry.iloc[0]

bg_mv_3310 = bg_3310[bg_3310.geometry.intersects(mv_polygon)].copy()
bg_mv_3310 = bg_mv_3310.reset_index(drop=True)
bg_mv_3310["centroid_3310"] = bg_mv_3310.geometry.centroid

OLD_COUNT = 240
print(f"Block groups (bounding-box method): {OLD_COUNT}")
print(f"Block groups (city-boundary method): {len(bg_mv_3310)}")

centroids_check = bg_mv_3310["centroid_3310"].to_crs(epsg=4326)
print(f"Centroid lat range:  {centroids_check.y.min():.4f} – {centroids_check.y.max():.4f}")
print(f"Centroid lon range: {centroids_check.x.min():.4f} – {centroids_check.x.max():.4f}")

# ================================
# STEP 3: Pull ACS data — B25044 (vehicles by tenure)
# ================================
import requests
import pandas as pd

url = (
    "https://api.census.gov/data/2022/acs/acs5"
    "?get=NAME,B01003_001E,B25044_004E,B25044_005E,B25044_010E,B25044_011E"
    "&for=block%20group:*"
    "&in=state:06%20county:085"
)
response = requests.get(url)
data = response.json()
df = pd.DataFrame(data[1:], columns=data[0])

numeric_cols = ["B01003_001E", "B25044_004E", "B25044_005E",
                "B25044_010E", "B25044_011E"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["GEOID"] = (
    df["state"].str.zfill(2) +
    df["county"].str.zfill(3) +
    df["tract"].str.zfill(6) +
    df["block group"].str.zfill(1)
)
df["vehicles"] = (
    (df["B25044_004E"] + df["B25044_010E"]) * 1 +
    (df["B25044_005E"] + df["B25044_011E"]) * 2
)
df.rename(columns={"B01003_001E": "population"}, inplace=True)
df = df[["GEOID", "population", "vehicles"]]
print("NaN check:", df["vehicles"].isna().sum(), "NaNs out of", len(df))

# ================================
# STEP 4: Pull ACS data — B25003 (tenure for renter share)
# ================================
url_tenure = (
    "https://api.census.gov/data/2022/acs/acs5"
    "?get=NAME,B25003_001E,B25003_002E,B25003_003E"
    "&for=block%20group:*"
    "&in=state:06%20county:085"
)
response_tenure = requests.get(url_tenure)
tenure_data = response_tenure.json()
df_tenure = pd.DataFrame(tenure_data[1:], columns=tenure_data[0])

for col in ["B25003_001E", "B25003_002E", "B25003_003E"]:
    df_tenure[col] = pd.to_numeric(df_tenure[col], errors="coerce")

df_tenure["GEOID"] = (
    df_tenure["state"].str.zfill(2) +
    df_tenure["county"].str.zfill(3) +
    df_tenure["tract"].str.zfill(6) +
    df_tenure["block group"].str.zfill(1)
)
df_tenure = df_tenure[["GEOID", "B25003_001E", "B25003_002E", "B25003_003E"]]
df_tenure.columns = ["GEOID", "total_units", "owner_units", "renter_units"]
print(f"Tenure NaN check: {df_tenure['renter_units'].isna().sum()} NaNs")

# ================================
# STEP 5: Merge ACS onto block groups
# ================================
bg_mv_merged = bg_mv_3310.merge(df, on="GEOID", how="inner")
bg_mv_merged = bg_mv_merged.merge(df_tenure, on="GEOID", how="left")
print(f"After merge: {len(bg_mv_merged)} block groups")

# ================================
# STEP 6: Centroids to lat/lon
# ================================
centroids_4326 = bg_mv_merged["centroid_3310"].to_crs(epsg=4326)
bg_mv_merged["lat"] = centroids_4326.y
bg_mv_merged["lon"] = centroids_4326.x

# ================================
# STEP 7: Demand scenarios
# ================================
EV_ADOPTION_RATE = 0.10
OWNER_RATE       = 0.05
RENTER_RATE      = 0.30

bg_mv_merged["renter_share"] = (
    bg_mv_merged["renter_units"] / bg_mv_merged["total_units"]
).fillna(0).clip(0, 1)

bg_mv_merged["owner_vehicles"]  = bg_mv_merged["vehicles"] * (1 - bg_mv_merged["renter_share"])
bg_mv_merged["renter_vehicles"] = bg_mv_merged["vehicles"] * bg_mv_merged["renter_share"]

# Scenario A: uniform 10% (base model)
bg_mv_merged["demand_A"] = bg_mv_merged["vehicles"] * EV_ADOPTION_RATE

# Scenario B: tenure-weighted public charging need
bg_mv_merged["demand_B"] = (
    bg_mv_merged["owner_vehicles"]  * OWNER_RATE +
    bg_mv_merged["renter_vehicles"] * RENTER_RATE
)

# Default demand used by MILP = Scenario A
bg_mv_merged["d_i"] = bg_mv_merged["demand_A"]

# ================================
# STEP 8: Final result table
# ================================
result = bg_mv_merged[[
    "GEOID", "lat", "lon", "population", "vehicles",
    "renter_share", "demand_A", "demand_B", "d_i"
]].copy()
result.columns = [
    "block_group", "lat", "lon", "population", "vehicles",
    "renter_share", "demand_A", "demand_B", "demand"
]
result = result.reset_index(drop=True)

print("\nSet I — Demand points (sample):")
print(result[["block_group","lat","lon","demand_A","demand_B","renter_share"]].head(10).to_string())
print(f"\nTotal |I| = {len(result)}")
print(f"\n── Demand Scenario Comparison ───────────────────────────────────")
print(f"  {'Metric':<40} {'Scenario A':>12} {'Scenario B':>12}")
print(f"  {'-'*64}")
print(f"  {'Total demand (units)':<40} {result['demand_A'].sum():>12.1f} {result['demand_B'].sum():>12.1f}")
print(f"  {'Mean per block group':<40} {result['demand_A'].mean():>12.1f} {result['demand_B'].mean():>12.1f}")
print(f"  {'Max per block group':<40} {result['demand_A'].max():>12.1f} {result['demand_B'].max():>12.1f}")
print(f"  {'Mean renter share':<40} {result['renter_share'].mean():>12.3f} {'(weighted)':>12}")
print(f"────────────────────────────────────────────────────────────────")

top_renters = result.nlargest(10, "renter_share")[
    ["block_group", "renter_share", "demand_A", "demand_B"]
].copy()
top_renters["demand_shift"] = top_renters["demand_B"] - top_renters["demand_A"]
print(f"\nTop 10 block groups by renter share:")
print(top_renters.to_string(index=False))

# ================================
# STEP 9: Geography audit (filled after distance matrix)
# ================================
OLD_DEMAND = 15663.6
print(f"\n── Geography Audit (partial — complete after dist. matrix) ─────")
print(f"  Block groups: {OLD_COUNT} (bbox) → {len(result)} (city boundary)")
print(f"  Total demand (Scenario A): {OLD_DEMAND} → {result['demand_A'].sum():.1f}")
print(f"────────────────────────────────────────────────────────────────")

# ================================
# STEP 10: Save
# ================================
result.to_csv("Mountain_View_demand_points.csv", index=False)
print("\nSaved to Mountain_View_demand_points.csv")

## 2. Home-Charging Vulnerability Index (HCVI)
Builds a four-component composite index (renter share, income vulnerability, multifamily housing share, zero-vehicle household share) to identify block groups most dependent on public charging infrastructure. Defines high-vulnerability set H as the top quartile.

In [ ]:
# ================================
Home-Charging Vulnerability Index (HCVI)
# ================================
import requests
import pandas as pd
import numpy as np

# ================================
# STEP 1: Pull additional ACS variables
# B19013_001E = median household income
# B25024_001E = total housing units
# B25024_002E = 1-unit detached (single family — easiest home charging)
# B25044_003E = owner-occupied, no vehicle
# B25044_009E = renter-occupied, no vehicle
# ================================
url_hcvi = (
    "https://api.census.gov/data/2022/acs/acs5"
    "?get=NAME,B19013_001E,B25024_001E,B25024_002E,"
    "B25044_003E,B25044_009E"
    "&for=block%20group:*"
    "&in=state:06%20county:085"
)
response_hcvi = requests.get(url_hcvi)
hcvi_data = response_hcvi.json()
df_hcvi = pd.DataFrame(hcvi_data[1:], columns=hcvi_data[0])

hcvi_cols = ["B19013_001E", "B25024_001E", "B25024_002E",
             "B25044_003E", "B25044_009E"]
for col in hcvi_cols:
    df_hcvi[col] = pd.to_numeric(df_hcvi[col], errors="coerce")

df_hcvi["GEOID"] = (
    df_hcvi["state"].str.zfill(2) +
    df_hcvi["county"].str.zfill(3) +
    df_hcvi["tract"].str.zfill(6) +
    df_hcvi["block group"].str.zfill(1)
)
df_hcvi = df_hcvi[[
    "GEOID", "B19013_001E", "B25024_001E",
    "B25024_002E", "B25044_003E", "B25044_009E"
]]
df_hcvi.columns = [
    "GEOID", "median_income", "total_housing",
    "single_family_units", "owner_no_vehicle", "renter_no_vehicle"
]

print(f"HCVI data NaN check:")
print(f"  median_income NaNs:      {df_hcvi['median_income'].isna().sum()}")
print(f"  single_family NaNs:      {df_hcvi['single_family_units'].isna().sum()}")

# ================================
# STEP 2: Merge onto result
# ================================
# Bring total_units in from the tenure data already pulled in Cell 1
result_with_tenure = result.copy()
result_with_tenure["total_units"] = bg_mv_merged["total_units"].values

result_hcvi = result_with_tenure.merge(
    df_hcvi, left_on="block_group", right_on="GEOID", how="left"
)

# ================================
# STEP 3: Compute component scores (each normalized to [0,1])
# Higher score = more vulnerable
# ================================

# Component 1: Renter share (already computed)
# Higher renter share = more vulnerable
result_hcvi["c1_renter"] = result_hcvi["renter_share"]

# Component 2: Income vulnerability
# Lower income = more vulnerable; invert and normalize
# Replace -666666666 (ACS code for missing/unreliable) with NaN
result_hcvi["median_income"] = result_hcvi["median_income"].replace(
    -666666666, np.nan
)
inc_min = result_hcvi["median_income"].min()
inc_max = result_hcvi["median_income"].max()
result_hcvi["c2_income"] = (
    1 - (result_hcvi["median_income"] - inc_min) / (inc_max - inc_min)
).fillna(0.5)  # fill missing with median vulnerability

# Component 3: Multifamily share
# Higher multifamily (lower single-family) = more vulnerable
result_hcvi["single_family_share"] = (
    result_hcvi["single_family_units"] / result_hcvi["total_housing"]
).fillna(0).clip(0, 1)
result_hcvi["c3_multifamily"] = 1 - result_hcvi["single_family_share"]

# Component 4: Zero-vehicle household share
# Higher zero-vehicle share = more transit dependent = more vulnerable
result_hcvi["zero_vehicle_units"] = (
    result_hcvi["owner_no_vehicle"] + result_hcvi["renter_no_vehicle"]
)
result_hcvi["zero_vehicle_share"] = (
    result_hcvi["zero_vehicle_units"] / result_hcvi["total_units"]
).fillna(0).clip(0, 1)
result_hcvi["c4_zero_vehicle"] = result_hcvi["zero_vehicle_share"]

# ================================
# STEP 4: Composite HCVI (equal-weighted average)
# ================================
components = ["c1_renter", "c2_income", "c3_multifamily", "c4_zero_vehicle"]
result_hcvi["HCVI"] = result_hcvi[components].mean(axis=1)

# Normalize HCVI to [0,1]
hcvi_min = result_hcvi["HCVI"].min()
hcvi_max = result_hcvi["HCVI"].max()
result_hcvi["HCVI_norm"] = (
    (result_hcvi["HCVI"] - hcvi_min) / (hcvi_max - hcvi_min)
)

# ================================
# STEP 5: Define high-vulnerability set H
# H = top quartile by HCVI score
# ================================
hcvi_threshold = result_hcvi["HCVI_norm"].quantile(0.75)
result_hcvi["high_vulnerability"] = result_hcvi["HCVI_norm"] >= hcvi_threshold

H_idx = result_hcvi.index[result_hcvi["high_vulnerability"]].tolist()
H_set = set(H_idx)

print(f"\n── Home-Charging Vulnerability Index ───────────────────────────")
print(f"  Block groups total:           {len(result_hcvi)}")
print(f"  High-vulnerability set |H|:   {len(H_idx)} (top quartile)")
print(f"  HCVI threshold (75th pct):    {hcvi_threshold:.3f}")
print(f"\n  Component means:")
for c in components:
    print(f"    {c}: {result_hcvi[c].mean():.3f}")
print(f"\n  HCVI stats:")
print(f"    Mean:   {result_hcvi['HCVI_norm'].mean():.3f}")
print(f"    Median: {result_hcvi['HCVI_norm'].median():.3f}")
print(f"    Min:    {result_hcvi['HCVI_norm'].min():.3f}")
print(f"    Max:    {result_hcvi['HCVI_norm'].max():.3f}")

# ================================
# STEP 6: Top 10 most vulnerable block groups
# ================================
top_vulnerable = result_hcvi.nlargest(10, "HCVI_norm")[[
    "block_group", "HCVI_norm", "renter_share",
    "c2_income", "c3_multifamily", "c4_zero_vehicle"
]].copy()
print(f"\nTop 10 most vulnerable block groups:")
print(top_vulnerable.round(3).to_string(index=False))

# ================================
# STEP 7: Evaluate HCVI coverage under MILP solution
# ================================
d_A = result["demand_A"].values

# Check which H block groups are served by MILP solution
print(f"\n── HCVI Coverage under MILP Solution (Scenario A) ──────────────")

# Rebuild assignment from solution_df
milp_assignment = {}
for _, srow in solution_df.iterrows():
    j_idx = sites[sites["site_id"] == srow["site_id"]].index[0]
    for i in range(len(result)):
        if D[i, j_idx] <= MAX_DIST_KM:
            # Approximate: assign block group to nearest open station
            pass

# Simpler: use neighbors and solution
open_site_ids = set(solution_df["site_id"])
open_site_indices = [
    sites[sites["site_id"] == sid].index[0]
    for sid in open_site_ids
]

# For each block group, check if any open site is within 3km
served_in_H   = sum(
    1 for i in H_idx
    if any(D[i, j] <= MAX_DIST_KM for j in open_site_indices)
)
unserved_in_H = len(H_idx) - served_in_H

served_out_H   = sum(
    1 for i in range(len(result)) if i not in H_set
    and any(D[i, j] <= MAX_DIST_KM for j in open_site_indices)
)

# Average distance for H vs non-H (use nearest open site as proxy)
def nearest_open_dist(i):
    dists = [D[i, j] for j in open_site_indices if D[i, j] <= MAX_DIST_KM]
    return min(dists) if dists else np.nan

avg_dist_H     = np.nanmean([nearest_open_dist(i) for i in H_idx])
avg_dist_non_H = np.nanmean([
    nearest_open_dist(i) for i in range(len(result)) if i not in H_set
])

demand_H     = sum(d_A[i] for i in H_idx)
demand_non_H = sum(d_A[i] for i in range(len(result)) if i not in H_set)

print(f"  High-vulnerability block groups |H|:  {len(H_idx)}")
print(f"  H served (within 3km of open site):   {served_in_H} / {len(H_idx)}")
print(f"  H unserved:                            {unserved_in_H} / {len(H_idx)}")
print(f"  Non-H served:                          {served_out_H} / {len(result)-len(H_idx)}")
print(f"  Avg nearest distance H:                {avg_dist_H:.3f} km")
print(f"  Avg nearest distance non-H:            {avg_dist_non_H:.3f} km")
print(f"  Distance gap (H - non-H):              {avg_dist_H - avg_dist_non_H:.3f} km")
print(f"  EV demand in H:                        {demand_H:.1f} units")
print(f"  EV demand in non-H:                    {demand_non_H:.1f} units")
print(f"────────────────────────────────────────────────────────────────")

# ================================
# STEP 8: Save
# ================================
result_hcvi[[
    "block_group", "lat", "lon", "HCVI_norm",
    "c1_renter", "c2_income", "c3_multifamily", "c4_zero_vehicle",
    "high_vulnerability"
]].to_csv("Mountain_View_HCVI.csv", index=False)
print("\nSaved to Mountain_View_HCVI.csv")

# Make H_idx and H_set available for Fix 8
print(f"\nH_idx defined: {len(H_idx)} block groups in high-vulnerability set")

## 3. Candidate Sites — OSM Parking Lots
Queries OpenStreetMap for parking lots within Mountain View's administrative boundary. Returns 792 candidate sites.

In [ ]:
!pip install osmnx -q
import osmnx as ox
import geopandas as gpd
import pandas as pd

place = "Mountain View, California, USA"
tags  = {"amenity": "parking"}
parking = ox.features_from_place(place, tags=tags)

print(f"Raw OSM features found: {len(parking)}")
print(parking.geometry.geom_type.value_counts())

parking = parking[["geometry"]].copy()
parking = parking.to_crs(epsg=3310)
parking["centroid"] = parking.geometry.centroid
parking["lat"] = parking["centroid"].to_crs(epsg=4326).y
parking["lon"] = parking["centroid"].to_crs(epsg=4326).x
parking = parking.dropna(subset=["lat", "lon"]).reset_index(drop=True)

FIXED_COST = 50000
parking["site_id"] = ["J" + str(i) for i in range(len(parking))]
parking["c_j"]     = FIXED_COST

sites = parking[["site_id", "lat", "lon", "c_j"]].copy()
print(f"\nSet J — Candidate sites (sample):")
print(sites.head(10).to_string())
print(f"\nTotal candidate sites |J| = {len(sites)}")

sites.to_csv("Mountain_View_candidate_sites.csv", index=False)
print("Saved to Mountain_View_candidate_sites.csv")

## 4. Candidate Site Feasibility Tiers
Classifies the 792 OSM parking lots into five feasibility tiers based on access, operator, and parking-type attributes. Identifies implementation risks among MILP-selected stations.

In [ ]:
# ================================
Candidate site feasibility tiers
# ================================
import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np

# ================================
# STEP 1: Pull richer OSM attributes for parking lots
# ================================
place = "Mountain View, California, USA"
tags  = {"amenity": "parking"}

parking_rich = ox.features_from_place(place, tags=tags)

# Keep useful OSM attributes
keep_cols = [c for c in [
    "geometry", "access", "operator", "operator:type",
    "parking", "fee", "name", "owned_by"
] if c in parking_rich.columns]

parking_rich = parking_rich[keep_cols].copy()
parking_rich = parking_rich.to_crs(epsg=3310)
parking_rich["centroid"] = parking_rich.geometry.centroid
parking_rich["lat"] = parking_rich["centroid"].to_crs(epsg=4326).y
parking_rich["lon"] = parking_rich["centroid"].to_crs(epsg=4326).x
parking_rich = parking_rich.dropna(subset=["lat", "lon"]).reset_index(drop=True)
parking_rich["site_id"] = ["J" + str(i) for i in range(len(parking_rich))]

print(f"Total candidate sites: {len(parking_rich)}")
print(f"\nAccess values:")
if "access" in parking_rich.columns:
    print(parking_rich["access"].value_counts(dropna=False).to_string())
print(f"\nParking type values:")
if "parking" in parking_rich.columns:
    print(parking_rich["parking"].value_counts(dropna=False).to_string())
print(f"\nOperator type values:")
if "operator:type" in parking_rich.columns:
    print(parking_rich["operator:type"].value_counts(dropna=False).to_string())

# ================================
# STEP 2: Assign feasibility tiers
# ================================
def assign_tier(row):
    access   = str(row.get("access", "")).lower()
    parking  = str(row.get("parking", "")).lower()
    operator = str(row.get("operator", "")).lower()
    op_type  = str(row.get("operator:type", "")).lower()
    name     = str(row.get("name", "")).lower()

    # Tier 5: Private/restricted — lowest viability
    if access in ["private", "no", "permissive"]:
        return 5

    # Tier 4: Underground/structure — high installation cost
    if parking in ["underground", "multi-storey", "garage"]:
        return 4

    # Tier 1: Municipal/government — highest viability
    if op_type in ["government", "public"] or "city" in operator or "municipality" in operator:
        return 1

    # Tier 2: Retail/commercial — good viability
    if any(kw in operator for kw in ["walmart", "target", "safeway", "costco",
                                      "whole foods", "trader joe"]):
        return 2
    if any(kw in name for kw in ["shopping", "mall", "plaza", "center", "market"]):
        return 2

    # Tier 3: School/faith/HOA — moderate viability, negotiation needed
    if any(kw in name for kw in ["school", "church", "temple", "mosque",
                                  "faith", "elementary", "middle", "high school"]):
        return 3
    if any(kw in operator for kw in ["school", "church", "temple"]):
        return 3

    # Default: surface lot, unknown operator — moderate viability
    return 3

parking_rich["tier"] = parking_rich.apply(assign_tier, axis=1)

tier_labels = {
    1: "Municipal/government",
    2: "Retail/commercial",
    3: "Surface lot / school / faith (unknown operator)",
    4: "Underground/structured parking",
    5: "Private/restricted access"
}

# ================================
# STEP 3: Summary
# ================================
print(f"\n── Candidate Site Feasibility Tiers ─────────────────────────────")
print(f"  {'Tier':<6} {'Label':<48} {'Count':>6} {'%':>6}")
print(f"  {'-'*66}")
tier_counts = parking_rich["tier"].value_counts().sort_index()
for tier, count in tier_counts.items():
    pct = count / len(parking_rich) * 100
    print(f"  {tier:<6} {tier_labels[tier]:<48} {count:>6} {pct:>5.1f}%")
print(f"  {'-'*66}")
print(f"  {'Total':<54} {len(parking_rich):>6}")
print(f"────────────────────────────────────────────────────────────────")

# ================================
# STEP 4: Check which MILP-selected sites fall in each tier
# ================================
# Match solution_df site IDs to tiers
print(f"\n── MILP-Selected Station Tiers ──────────────────────────────────")
selected_ids = set(solution_df["site_id"])
selected_tiers = []
for sid in selected_ids:
    idx = int(sid[1:])  # strip "J" prefix
    if idx < len(parking_rich):
        t = parking_rich["tier"].iloc[idx]
        selected_tiers.append((sid, t, tier_labels[t]))

selected_tiers.sort(key=lambda x: x[1])
tier_summary = pd.DataFrame(selected_tiers, columns=["site_id", "tier", "label"])
tier_dist = tier_summary["tier"].value_counts().sort_index()

for tier, count in tier_dist.items():
    print(f"  Tier {tier} ({tier_labels[tier]}): {count} stations")
print(f"────────────────────────────────────────────────────────────────")

# ================================
# STEP 5: Save
# ================================
parking_rich[["site_id", "lat", "lon", "tier"]].to_csv(
    "Mountain_View_site_tiers.csv", index=False
)
print(f"\nSaved to Mountain_View_site_tiers.csv")

## 5. Distance Matrix
Computes pairwise haversine distances between all 79 demand centroids and 792 candidate sites. Applies 3 km proximity filter, reducing from 62,568 to 30,997 valid pairs.

In [ ]:
import numpy as np
import pandas as pd

I = result[["block_group", "lat", "lon"]].copy()
J = sites[["site_id", "lat", "lon"]].copy()

def haversine_matrix(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2[:, None] - lat1[None, :]
    dlon = lon2[:, None] - lon1[None, :]
    a = (np.sin(dlat/2)**2 +
         np.cos(lat1[None, :]) * np.cos(lat2[:, None]) * np.sin(dlon/2)**2)
    return 2 * R * np.arcsin(np.sqrt(a))

D = haversine_matrix(
    I["lat"].values, I["lon"].values,
    J["lat"].values, J["lon"].values
).T  # shape: (|I|, |J|)

print(f"Distance matrix shape: {D.shape}")
print(f"Min: {D.min():.4f} km  Max: {D.max():.4f} km  Mean: {D.mean():.4f} km")

MAX_DIST_KM = 3.0
valid_pairs     = [(i, j) for i in range(len(I))
                   for j in range(len(J)) if D[i, j] <= MAX_DIST_KM]
reachable_mask  = [(D[i, :] <= MAX_DIST_KM).any() for i in range(len(I))]
n_reachable     = sum(reachable_mask)
n_unreachable   = len(I) - n_reachable

print(f"\nValid pairs within {MAX_DIST_KM} km: {len(valid_pairs)}")
print(f"Reduction from full matrix: {(1 - len(valid_pairs)/(len(I)*len(J)))*100:.1f}%")

print(f"\n── Geography Audit (Complete) ──────────────────────────────────────")
print(f"  {'Metric':<45} {'Old (bbox)':>12} {'New (boundary)':>15}")
print(f"  {'-'*72}")
print(f"  {'Block groups |I|':<45} {240:>12} {len(I):>15}")
print(f"  {'Total EV demand (Scenario A)':<45} {15663.6:>12.1f} {result['demand_A'].sum():>15.1f}")
print(f"  {'Total EV demand (Scenario B)':<45} {'—':>12} {result['demand_B'].sum():>15.1f}")
print(f"  {'Valid pairs within 3 km':<45} {37300:>12} {len(valid_pairs):>15}")
print(f"  {'Reachable block groups |I*|':<45} {143:>12} {n_reachable:>15}")
print(f"  {'Unreachable block groups':<45} {97:>12} {n_unreachable:>15}")
print(f"  {'-'*72}")

D_df = pd.DataFrame(D, index=I["block_group"], columns=J["site_id"])
D_df.to_csv("Mountain_View_distance_matrix.csv")
print("\nSaved to Mountain_View_distance_matrix.csv")

## 6. MILP Solver — Scenario A (Uniform 10% Adoption)
Solves the Capacitated Facility Location MILP using OR-Tools SCIP. Objective: minimize demand-weighted travel distance. Enforces exactly K=20 stations, Q=500 capacity per station, 3 km proximity constraint. Includes full solver audit diagnostics.

In [ ]:
!pip install -q ortools

from ortools.linear_solver import pywraplp
import numpy as np
import time

K         = 20
CAP       = 500
FIXED_COST = 50000

num_I = len(result)
num_J = len(sites)
I_idx = range(num_I)
J_idx = range(num_J)

# Use Scenario A demand (uniform 10%)
d = result["demand_A"].values

MAX_DIST_KM = 3.0

neighbors_of_i = {i: [j for j in J_idx if D[i, j] <= MAX_DIST_KM] for i in I_idx}
neighbors_of_j = {j: [i for i in I_idx if D[i, j] <= MAX_DIST_KM] for j in J_idx}
valid_pairs    = [(i, j) for i in I_idx for j in neighbors_of_i[i]]
reachable      = [i for i in I_idx if neighbors_of_i[i]]
unreachable    = [i for i in I_idx if not neighbors_of_i[i]]

print(f"Valid pairs: {len(valid_pairs)}")
print(f"Reachable: {len(reachable)} / {num_I}")
print(f"Unreachable: {len(unreachable)}")

solver = pywraplp.Solver.CreateSolver("SCIP")
solver.SetTimeLimit(300_000)

x = [solver.BoolVar(f"x_{j}") for j in J_idx]
y = {(i, j): solver.BoolVar(f"y_{i}_{j}") for (i, j) in valid_pairs}
u = [solver.BoolVar(f"u_{i}") for i in I_idx]

M_penalty = float(MAX_DIST_KM * max(d))
objective  = solver.Objective()
for (i, j) in valid_pairs:
    objective.SetCoefficient(y[i, j], float(d[i]) * float(D[i, j]))
for i in I_idx:
    objective.SetCoefficient(u[i], M_penalty * float(d[i]))
objective.SetMinimization()

for i in I_idx:
    if neighbors_of_i[i]:
        solver.Add(sum(y[i, j] for j in neighbors_of_i[i]) + u[i] == 1)
    else:
        solver.Add(u[i] == 1)

for (i, j) in valid_pairs:
    solver.Add(y[i, j] <= x[j])

solver.Add(sum(x[j] for j in J_idx) == K)

for j in J_idx:
    if neighbors_of_j[j]:
        solver.Add(sum(d[i] * y[i, j] for i in neighbors_of_j[j]) <= CAP)

print(f"Variables:   {solver.NumVariables()}")
print(f"Constraints: {solver.NumConstraints()}")
print("Solving (Scenario A)...")

t0     = time.time()
status = solver.Solve()
solve_time = time.time() - t0


# ================================
Solver audit diagnostics
# ================================
print(f"\n── Solver Audit ─────────────────────────────────────────────────")
print(f"  Solver:              OR-Tools SCIP v9.x")
print(f"  Variables:           {solver.NumVariables()} (all binary)")
print(f"  Constraints:         {solver.NumConstraints()}")
print(f"  Status:              {'Optimal' if status == pywraplp.Solver.OPTIMAL else 'Feasible'}")
print(f"  Objective value:     {solver.Objective().Value():,.4f} EV-vehicle·km")
print(f"  Best bound:          {solver.Objective().BestBound():,.4f} EV-vehicle·km")
if solver.Objective().BestBound() > 0:
    mip_gap = abs(solver.Objective().Value() - solver.Objective().BestBound()) / abs(solver.Objective().Value())
    print(f"  MIP gap:             {mip_gap*100:.4f}%")
else:
    print(f"  MIP gap:             0.0000% (proven optimal)")
print(f"  Solve time:          {solve_time:.2f}s")
print(f"  Time limit:          300s")
print(f"  Hardware:            Google Colab (Intel Xeon, single thread)")
print(f"────────────────────────────────────────────────────────────────")

# ================================
#
# ================================


if status in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE]:
    print(f"\nStatus: {'Optimal' if status == pywraplp.Solver.OPTIMAL else 'Feasible'}")

    open_sites      = [j for j in J_idx if x[j].solution_value() > 0.5]
    unserved_idx    = [i for i in I_idx if u[i].solution_value() > 0.5]
    unserved_demand = sum(d[i] for i in unserved_idx)
    total_demand    = sum(d)
    served_demand   = total_demand - unserved_demand

    travel_cost = sum(
        d[i] * D[i, j] * y[i, j].solution_value()
        for (i, j) in valid_pairs
    )

    print(f"Stations built:   {len(open_sites)} / {K}")
    print(f"Unserved:         {len(unserved_idx)} / {num_I}")
    print(f"Served demand:    {served_demand:.1f} / {total_demand:.1f}")
    print(f"Fixed cost:       ${len(open_sites) * FIXED_COST:,.0f}")
    print(f"Travel cost:      {travel_cost:,.2f} EV-vehicle·km")

    rows = []
    for j in open_sites:
        assigned   = [i for i in neighbors_of_j[j] if y[i, j].solution_value() > 0.5]
        stn_demand = sum(d[i] for i in assigned)
        avg_dist   = (sum(D[i, j] for i in assigned) / len(assigned)
                      if assigned else 0.0)
        rows.append({
            "site_id":      sites["site_id"].iloc[j],
            "lat":          sites["lat"].iloc[j],
            "lon":          sites["lon"].iloc[j],
            "n_assigned":   len(assigned),
            "stn_demand":   round(stn_demand, 1),
            "avg_dist_km":  round(avg_dist, 3),
            "load_pct":     round(stn_demand / CAP * 100, 1)
        })

    solution_df = pd.DataFrame(rows)
    print("\nSelected stations:")
    print(solution_df.to_string(index=False))

    avg_assign_dist = travel_cost / served_demand if served_demand > 0 else 0
    avg_load        = served_demand / len(open_sites) if open_sites else 0

    print(f"\nAvg. assignment distance: {avg_assign_dist:.3f} km")
    print(f"Avg. station load:        {avg_load:.1f} EV units ({avg_load/CAP*100:.1f}% of cap)")
    print(f"Solve time:               {solve_time:.1f}s")

    solution_df.to_csv("Mountain_View_CFLP_solution.csv", index=False)
    print("\nSaved to Mountain_View_CFLP_solution.csv")

elif status == pywraplp.Solver.INFEASIBLE:
    print("INFEASIBLE")
else:
    print(f"Status code: {status}")

## 7. MILP Solver — Scenario B (Tenure-Weighted Demand)
Re-solves the MILP under tenure-weighted demand (renters 30%, owners 5%). Total demand rises to 10,518 units, exceeding network capacity. Includes solver audit and site selection comparison vs Scenario A.

In [ ]:
# ================================
# MILP — Scenario B (tenure-weighted demand)
# Reuses neighbor dicts from Cell 4 (distance matrix unchanged)
# ================================
!pip install -q ortools
from ortools.linear_solver import pywraplp
import numpy as np
import time

K          = 20
CAP        = 500
MAX_DIST_KM = 3.0

# Scenario B demand vector
d_B = result["demand_B"].values

print(f"Scenario A total demand: {result['demand_A'].sum():.1f} units")
print(f"Scenario B total demand: {result['demand_B'].sum():.1f} units")
print(f"Mean renter share: {result['renter_share'].mean():.3f}")

# Reuse neighbor dicts — only d changes, not D
# (neighbors_of_i and neighbors_of_j already defined in Cell 4)
valid_pairs_B = [(i, j) for i in I_idx for j in neighbors_of_i[i]]

solver_B = pywraplp.Solver.CreateSolver("SCIP")
solver_B.SetTimeLimit(300_000)

x_B = [solver_B.BoolVar(f"x_{j}") for j in J_idx]
y_B = {(i, j): solver_B.BoolVar(f"y_{i}_{j}") for (i, j) in valid_pairs_B}
u_B = [solver_B.BoolVar(f"u_{i}") for i in I_idx]

M_B   = float(MAX_DIST_KM * max(d_B))
obj_B = solver_B.Objective()
for (i, j) in valid_pairs_B:
    obj_B.SetCoefficient(y_B[i, j], float(d_B[i]) * float(D[i, j]))
for i in I_idx:
    obj_B.SetCoefficient(u_B[i], M_B * float(d_B[i]))
obj_B.SetMinimization()

for i in I_idx:
    if neighbors_of_i[i]:
        solver_B.Add(sum(y_B[i, j] for j in neighbors_of_i[i]) + u_B[i] == 1)
    else:
        solver_B.Add(u_B[i] == 1)

for (i, j) in valid_pairs_B:
    solver_B.Add(y_B[i, j] <= x_B[j])

solver_B.Add(sum(x_B[j] for j in J_idx) == K)

for j in J_idx:
    if neighbors_of_j[j]:
        solver_B.Add(sum(d_B[i] * y_B[i, j] for i in neighbors_of_j[j]) <= CAP)

print(f"\nVariables:   {solver_B.NumVariables()}")
print(f"Constraints: {solver_B.NumConstraints()}")
print("Solving (Scenario B)...")

t0_B     = time.time()
solverParams = pywraplp.MPSolverParameters()
solverParams.SetDoubleParam(
    pywraplp.MPSolverParameters.RELATIVE_MIP_GAP, 0.01
)
status_B = solver_B.Solve(solverParams)
solve_B  = time.time() - t0_B

# ================================
Solver audit diagnostics
# ================================

print(f"\n── Solver Audit (Scenario B) ────────────────────────────────────")
print(f"  Variables:           {solver_B.NumVariables()} (all binary)")
print(f"  Constraints:         {solver_B.NumConstraints()}")
print(f"  Status:              {'Optimal' if status_B == pywraplp.Solver.OPTIMAL else 'Feasible (time limit)'}")
print(f"  Objective value:     {solver_B.Objective().Value():,.4f} EV-vehicle·km")
print(f"  Best bound:          {solver_B.Objective().BestBound():,.4f} EV-vehicle·km")
if solver_B.Objective().BestBound() > 0 and solver_B.Objective().Value() > 0:
    mip_gap_B = abs(solver_B.Objective().Value() - solver_B.Objective().BestBound()) / abs(solver_B.Objective().Value())
    print(f"  MIP gap:             {mip_gap_B*100:.2f}%")
print(f"  Solve time:          {solve_B:.2f}s")
print(f"  Time limit:          300s")
print(f"────────────────────────────────────────────────────────────────")

# ================================
#
# ================================

if status_B in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE]:
    print(f"\nStatus:    {'Optimal' if status_B == pywraplp.Solver.OPTIMAL else 'Feasible'}")
    print(f"Solve time: {solve_B:.1f}s")

    open_B     = [j for j in J_idx if x_B[j].solution_value() > 0.5]
    unserved_B = [i for i in I_idx if u_B[i].solution_value() > 0.5]
    travel_B   = sum(
        d_B[i] * D[i, j] * y_B[i, j].solution_value()
        for (i, j) in valid_pairs_B
    )
    served_dem_B = sum(d_B) - sum(d_B[i] for i in unserved_B)

    print(f"Stations built: {len(open_B)} / {K}")
    print(f"Unserved:       {len(unserved_B)} / {num_I}")
    print(f"Travel cost:    {travel_B:,.2f} EV-vehicle·km")
    if served_dem_B > 0:
        print(f"Avg distance:   {travel_B/served_dem_B:.3f} km")

    rows_B = []
    for j in open_B:
        assigned_B   = [i for i in neighbors_of_j[j]
                        if y_B[i, j].solution_value() > 0.5]
        stn_demand_B = sum(d_B[i] for i in assigned_B)
        avg_dist_B   = (sum(D[i, j] for i in assigned_B) / len(assigned_B)
                        if assigned_B else 0.0)
        rows_B.append({
            "site_id":     sites["site_id"].iloc[j],
            "lat":         sites["lat"].iloc[j],
            "lon":         sites["lon"].iloc[j],
            "n_assigned":  len(assigned_B),
            "stn_demand":  round(stn_demand_B, 1),
            "avg_dist_km": round(avg_dist_B, 3),
            "load_pct":    round(stn_demand_B / CAP * 100, 1)
        })

    print(f"Total served demand: {served_dem_B:.1f} units")
    print(f"Avg station load: {served_dem_B/len(open_B):.1f} units ({served_dem_B/len(open_B)/CAP*100:.1f}%)")

    solution_B_df = pd.DataFrame(rows_B)
    print("\nSelected stations (Scenario B):")
    print(solution_B_df.to_string(index=False))

    # Compare site selection A vs B
    sites_A = set(solution_df["site_id"])
    sites_B = set(solution_B_df["site_id"])
    print(f"\n── Site Selection Comparison A vs B ─────────────────────────────")
    print(f"  Sites in both:      {len(sites_A & sites_B)} / {K}")
    print(f"  Only in Scenario A: {sorted(sites_A - sites_B)}")
    print(f"  Only in Scenario B: {sorted(sites_B - sites_A)}")
    print(f"────────────────────────────────────────────────────────────────")

    solution_B_df.to_csv("Mountain_View_CFLP_solution_B.csv", index=False)
    print("\nSaved to Mountain_View_CFLP_solution_B.csv")

else:
    print(f"Status code: {status_B}")

## 8. Heuristic — Coverage-First Greedy + Simulated Annealing
Two-phase solver-free heuristic. Phase 1: iteratively opens the site covering the most uncovered block groups. Phase 2: simulated annealing with geometric cooling (T₀=500, α=0.995, 55s runtime). Achieves within 1.6% of optimal.

In [ ]:
import numpy as np
import time
import math

K        = 20
CAP      = 500
MAX_DIST = 3.0
FIXED_COST = 50000
SA_TIME  = 55
T_START  = 500.0
T_END    = 0.1
ALPHA    = 0.995

d       = result["demand_A"].values
num_I   = len(d)
num_J   = len(sites)
I_idx   = list(range(num_I))
J_idx   = list(range(num_J))

neighbors_of_i = {i: [j for j in J_idx if D[i, j] <= MAX_DIST] for i in I_idx}
neighbors_of_j = {j: [i for i in I_idx if D[i, j] <= MAX_DIST] for j in J_idx}
reachable      = [i for i in I_idx if neighbors_of_i[i]]

def assign_coverage(open_set):
    load       = {j: 0.0 for j in open_set}
    assignment = {}
    for i in sorted(reachable, key=lambda i: -d[i]):
        candidates = [j for j in neighbors_of_i[i]
                      if j in open_set and load[j] + d[i] <= CAP]
        if not candidates:
            candidates = [j for j in neighbors_of_i[i] if j in open_set]
        if candidates:
            j = min(candidates, key=lambda j: D[i, j])
            assignment[i] = j
            load[j]      += d[i]
    return assignment, load

def objective(assignment):
    return sum(d[i] * D[i, assignment[i]] for i in assignment)

# Phase 1: Coverage-first greedy
print("Phase 1: Coverage-first greedy...")
t0        = time.time()
uncovered = set(reachable)
open_set  = set()
closed_set = set(J_idx)

while len(open_set) < K and uncovered:
    best_j, best_cover = None, -1
    for j in closed_set:
        cover = sum(1 for i in neighbors_of_j[j] if i in uncovered)
        if cover > best_cover:
            best_cover, best_j = cover, j
    if best_j is None or best_cover == 0:
        break
    open_set.add(best_j)
    closed_set.remove(best_j)
    uncovered -= set(neighbors_of_j[best_j])

if len(open_set) < K:
    def density(j):
        nb = neighbors_of_j[j]
        return sum(d[i] for i in nb) / (np.mean([D[i,j] for i in nb]) + 1e-6) if nb else 0
    for j in sorted(closed_set, key=lambda j: -density(j)):
        if len(open_set) >= K:
            break
        open_set.add(j)
        closed_set.discard(j)

assignment, load = assign_coverage(open_set)
obj = objective(assignment)
print(f"  Covered: {len(assignment)}/{len(reachable)}  Obj: {obj:,.2f}  Time: {time.time()-t0:.2f}s")

# Phase 2: Simulated annealing
print("Phase 2: Simulated annealing...")
best_open, best_assignment, best_obj = set(open_set), dict(assignment), obj
current_open, current_assignment, current_obj = set(open_set), dict(assignment), obj
T, n_iter, n_accepted, n_improved = T_START, 0, 0, 0
closed_list = list(closed_set)
sa_start = time.time()

while time.time() - sa_start < SA_TIME:
    j_out = np.random.choice(list(current_open))
    j_in  = np.random.choice(closed_list)
    candidate = (current_open - {j_out}) | {j_in}
    new_assignment, _ = assign_coverage(candidate)
    if len(new_assignment) < len(reachable):
        T = max(T * ALPHA, T_END)
        n_iter += 1
        continue
    new_obj = objective(new_assignment)
    delta   = new_obj - current_obj
    if delta < 0 or np.random.random() < math.exp(-delta / T):
        current_open, current_assignment, current_obj = candidate, new_assignment, new_obj
        closed_list = [j for j in J_idx if j not in current_open]
        n_accepted += 1
        if current_obj < best_obj:
            best_open, best_assignment, best_obj = set(current_open), dict(current_assignment), current_obj
            n_improved += 1
    T = max(T * ALPHA, T_END)
    n_iter += 1

total_time = time.time() - t0
print(f"  Iterations: {n_iter:,}  Accepted: {n_accepted:,}  Improved: {n_improved:,}")

# Results
print(f"\n{'='*50}")
print(f"HEURISTIC SOLUTION (Scenario A)")
print(f"{'='*50}")
print(f"Solve time:     {total_time:.2f}s")
print(f"Objective:      {best_obj:,.2f} EV-vehicle·km")
print(f"Stations built: {len(best_open)} / {K}")
print(f"Served:         {len(best_assignment)} / {len(reachable)} ({len(best_assignment)/len(reachable)*100:.1f}%)")

rows = []
for j in sorted(best_open):
    assigned   = [i for i in reachable if best_assignment.get(i) == j]
    stn_demand = sum(d[i] for i in assigned)
    avg_dist   = np.mean([D[i, j] for i in assigned]) if assigned else 0.0
    rows.append({
        "site_id":     sites["site_id"].iloc[j],
        "lat":         sites["lat"].iloc[j],
        "lon":         sites["lon"].iloc[j],
        "n_assigned":  len(assigned),
        "stn_demand":  round(stn_demand, 1),
        "avg_dist_km": round(avg_dist, 3),
        "load_pct":    round(stn_demand / CAP * 100, 1)
    })

heuristic_df = pd.DataFrame(rows)
print("\nSelected stations:")
print(heuristic_df.to_string(index=False))

served_demand_h = sum(d[i] for i in best_assignment)
print(f"\nAvg. assignment distance: {best_obj/served_demand_h:.3f} km")
print(f"Travel cost per served point: {best_obj/len(best_assignment):.2f} EV-vehicle·km")
print(f"\nMILP gap: +{(best_obj/len(best_assignment) - 2513.26/79)/( 2513.26/79)*100:.1f}%")

heuristic_df.to_csv("Mountain_View_heuristic_solution.csv", index=False)
print("\nSaved to Mountain_View_heuristic_solution.csv")

## 9. Existing Network Baseline + Incremental Expansion
Evaluates Mountain View's 33 publicly accessible EV stations (from AFDC database, May 2026). Identifies the single unserved block group and the best candidate site to close the coverage gap. Produces three-way comparison: existing vs existing+1 vs MILP.

In [ ]:
import pandas as pd
import numpy as np

existing_data = [
    ("Mountain View High School",                   37.360757, -122.065743),
    ("Computer History Museum - Tesla Supercharger", 37.415328, -122.076575),
    ("Holiday Inn Express - Tesla Destination",      37.389431, -122.092061),
    ("Hotel Vue - Tesla Destination",                37.381445, -122.074197),
    ("W. El Camino Real (2440)",                     37.398645, -122.108206),
    ("ZICO EVC Hotel Zico",                          37.379740, -122.068720),
    ("Essex Arlo (1030 Castro St)",                  37.385227, -122.084380),
    ("1328 W El Camino Real",                        37.388739, -122.089277),
    ("CBW Properties - W El Camino",                 37.389553, -122.092531),
    ("CBW Properties - Central Ave",                 37.396167, -122.075767),
    ("Hotel Strata",                                 37.381552, -122.075918),
    ("The Dean Apartments",                          37.404153, -122.112013),
    ("Walmart (600 Showers Dr)",                     37.400980, -122.108660),
    ("189 N Bernardo Ave",                           37.386278, -122.051469),
    ("East Middlefield Road (199)",                  37.397666, -122.062063),
    ("PF - Dana St",                                 37.386156, -122.065531),
    ("Best Buy - E El Camino Real",                  37.375758, -122.063126),
    ("Park Plaza Apartments",                        37.401167, -122.094361),
    ("Nob Hill Foods (1250 Grant Rd)",               37.378474, -122.075627),
    ("The Redwoods",                                 37.406439, -122.092526),
    ("Graham Middle School",                         37.381173, -122.084810),
    ("Imai Elementary",                              37.374313, -122.075145),
    ("Mistral Elementary",                           37.395283, -122.094086),
    ("Vargas Elementary",                            37.392987, -122.061715),
    ("Bubb Elementary",                              37.378234, -122.081814),
    ("Castro Elementary",                            37.394025, -122.092044),
    ("PF - Whisman (301 N Whisman Rd)",              37.397528, -122.058920),
    ("Ramada Hotel (55 Fairchild Dr)",               37.406022, -122.064005),
    ("El Monte Center - Tesla Supercharger",         37.389672, -122.095731),
    ("Mountain View Memory Care",                    37.386306, -122.085352),
    ("Monta Loma Plaza",                             37.411140, -122.093470),
    ("Palo Alto Plaza (541 Del Medio)",              37.404303, -122.115173),
    ("99 Market Ranch - Tesla Supercharger",         37.378052, -122.077578),
]

existing = pd.DataFrame(existing_data, columns=["station_name", "lat", "lon"])
print(f"Existing public EV stations: {len(existing)}")

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

d      = result["demand_A"].values
num_I  = len(d)
I_lats = result["lat"].values
I_lons = result["lon"].values

D_existing = np.array([
    [haversine(I_lats[i], I_lons[i], row["lat"], row["lon"])
     for _, row in existing.iterrows()]
    for i in range(num_I)
])

MAX_DIST     = 3.0
CAP_EXISTING = 500

assignment_existing = {}
load_existing       = {j: 0.0 for j in range(len(existing))}

for i in sorted(range(num_I), key=lambda i: -d[i]):
    candidates = [
        j for j in range(len(existing))
        if D_existing[i, j] <= MAX_DIST
        and load_existing[j] + d[i] <= CAP_EXISTING
    ]
    if candidates:
        j = min(candidates, key=lambda j: D_existing[i, j])
        assignment_existing[i] = j
        load_existing[j]      += d[i]

served_existing        = len(assignment_existing)
unserved_existing      = num_I - served_existing
served_demand_existing = sum(d[i] for i in assignment_existing)
total_demand           = sum(d)
travel_existing        = sum(
    d[i] * D_existing[i, assignment_existing[i]]
    for i in assignment_existing
)

print(f"\n── Existing Network Evaluation ─────────────────────────────────")
print(f"  Block groups served:   {served_existing} / {num_I}")
print(f"  Block groups unserved: {unserved_existing} / {num_I}")
print(f"  Served demand:         {served_demand_existing:.1f} / {total_demand:.1f} units")
print(f"  Travel cost:           {travel_existing:,.2f} EV-vehicle·km")
if served_existing > 0:
    print(f"  Avg distance:          {travel_existing/served_demand_existing:.3f} km")
print(f"────────────────────────────────────────────────────────────────")

# Unserved block group and best new site
unserved_bg = [i for i in range(num_I) if i not in assignment_existing]
print(f"\nUnserved block group(s): {len(unserved_bg)}")
for i in unserved_bg:
    print(f"  GEOID: {result['block_group'].iloc[i]}")
    print(f"  Demand: {d[i]:.1f} units")
    reachable_sites = sorted([
        (D[i, j], sites['site_id'].iloc[j])
        for j in range(len(sites)) if D[i, j] <= MAX_DIST
    ])
    best_dist, best_sid = reachable_sites[0]
    print(f"  Best candidate site: {best_sid} ({best_dist:.3f} km)")

# Three-way comparison
travel_plus1   = travel_existing + d[unserved_bg[0]] * reachable_sites[0][0]
served_plus1   = served_existing + 1
dem_plus1      = served_demand_existing + d[unserved_bg[0]]

print(f"\n── Three-Way Comparison ─────────────────────────────────────────")
print(f"  {'Metric':<35} {'Existing':>10} {'Exist+1':>10} {'MILP':>10}")
print(f"  {'-'*65}")
print(f"  {'Stations':<35} {'33':>10} {'34':>10} {'20':>10}")
print(f"  {'Block groups served':<35} {served_existing:>10} {served_plus1:>10} {79:>10}")
print(f"  {'Served demand (units)':<35} {served_demand_existing:>10.1f} {dem_plus1:>10.1f} {5339.4:>10.1f}")
print(f"  {'Travel cost (EV-veh·km)':<35} {travel_existing:>10.2f} {travel_plus1:>10.2f} {2513.26:>10.2f}")
print(f"  {'Avg distance (km)':<35} {travel_existing/served_demand_existing:>10.3f} {travel_plus1/dem_plus1:>10.3f} {0.471:>10.3f}")
print(f"  {'-'*65}")

existing.to_csv("Mountain_View_existing_stations.csv", index=False)
print("\nSaved to Mountain_View_existing_stations.csv")

## 10. Equity Analysis — HCVI-Stratified Coverage
Computes formal equity metrics (coverage gap, distance gap, D95) for both MILP and heuristic solutions, stratified by the high-vulnerability set H. Tests equity constraints of the form d̄_H ≤ d̄_{I\H} + δ for δ ∈ {0.0, 0.1, 0.2}.

In [ ]:
# ================================
Formal equity metrics + equity-constrained MILP
# ================================
from ortools.linear_solver import pywraplp
import numpy as np
import pandas as pd
import time

# ================================
# STEP 1: Formal equity metrics for MILP solution
# ================================
d_A = result["demand_A"].values

# Rebuild MILP assignment from solution_df
open_site_ids     = set(solution_df["site_id"])
open_site_indices = [
    sites[sites["site_id"] == sid].index[0]
    for sid in open_site_ids
]

# Assign each block group to nearest open station within 3km
milp_assignment = {}
for i in range(len(result)):
    candidates = [
        j for j in open_site_indices
        if D[i, j] <= MAX_DIST_KM
    ]
    if candidates:
        milp_assignment[i] = min(candidates, key=lambda j: D[i, j])

# Equity metrics
def equity_metrics(assignment, d, H_idx, label="Solution"):
    H_set    = set(H_idx)
    non_H    = [i for i in range(len(d)) if i not in H_set]

    served_H     = [i for i in H_idx   if i in assignment]
    served_nonH  = [i for i in non_H   if i in assignment]
    unserved_H   = [i for i in H_idx   if i not in assignment]
    unserved_nonH= [i for i in non_H   if i not in assignment]

    demand_H     = sum(d[i] for i in H_idx)
    demand_nonH  = sum(d[i] for i in non_H)

    avg_dist_H   = (np.mean([D[i, assignment[i]] for i in served_H])
                    if served_H else np.nan)
    avg_dist_nonH= (np.mean([D[i, assignment[i]] for i in served_nonH])
                    if served_nonH else np.nan)

    # D95: 95th percentile assignment distance across all served
    all_served    = list(assignment.keys())
    all_dists     = [D[i, assignment[i]] for i in all_served]
    d95           = np.percentile(all_dists, 95) if all_dists else np.nan

    # Coverage gap: fraction unserved in H minus fraction unserved in non-H
    coverage_gap  = (len(unserved_H)/len(H_idx)
                     - len(unserved_nonH)/len(non_H))

    # Distance gap: avg dist H minus avg dist non-H
    dist_gap      = avg_dist_H - avg_dist_nonH

    print(f"\n── Equity Metrics: {label} ──────────────────────────────────")
    print(f"  {'Metric':<40} {'H (vuln)':>10} {'Non-H':>10}")
    print(f"  {'-'*60}")
    print(f"  {'Block groups':<40} {len(H_idx):>10} {len(non_H):>10}")
    print(f"  {'Served':<40} {len(served_H):>10} {len(served_nonH):>10}")
    print(f"  {'Unserved':<40} {len(unserved_H):>10} {len(unserved_nonH):>10}")
    print(f"  {'EV demand (units)':<40} {demand_H:>10.1f} {demand_nonH:>10.1f}")
    print(f"  {'Avg assignment distance (km)':<40} {avg_dist_H:>10.3f} {avg_dist_nonH:>10.3f}")
    print(f"  {'-'*60}")
    print(f"  Coverage gap (H unserved% - nonH unserved%): {coverage_gap:+.3f}")
    print(f"  Distance gap (H dist - nonH dist):           {dist_gap:+.3f} km")
    print(f"  D95 (95th pct assignment distance):          {d95:.3f} km")
    print(f"────────────────────────────────────────────────────────────────")

    return {
        "served_H": len(served_H), "served_nonH": len(served_nonH),
        "unserved_H": len(unserved_H), "unserved_nonH": len(unserved_nonH),
        "avg_dist_H": avg_dist_H, "avg_dist_nonH": avg_dist_nonH,
        "coverage_gap": coverage_gap, "dist_gap": dist_gap, "D95": d95
    }

milp_eq = equity_metrics(milp_assignment, d_A, H_idx, "MILP (Scenario A)")

# Heuristic assignment
heu_open_ids     = set(heuristic_df["site_id"])
heu_open_indices = [
    sites[sites["site_id"] == sid].index[0]
    for sid in heu_open_ids
]
heu_assignment = {}
for i in range(len(result)):
    candidates = [
        j for j in heu_open_indices
        if D[i, j] <= MAX_DIST_KM
    ]
    if candidates:
        heu_assignment[i] = min(candidates, key=lambda j: D[i, j])

heu_eq = equity_metrics(heu_assignment, d_A, H_idx, "Heuristic (Scenario A)")

# ================================
# STEP 2: Equity-constrained MILP
# Vary delta: avg_dist_H <= avg_dist_nonH + delta
# delta = inf (unconstrained), 0.1, 0.0, -0.1 (force H closer than nonH)
# ================================
print(f"\n── Equity-Constrained MILP: Varying Delta ───────────────────────")
print(f"  Constraint: avg_dist(H) <= avg_dist(non-H) + delta")
print(f"  Unconstrained result: dist_gap = {milp_eq['dist_gap']:+.3f} km")
print(f"────────────────────────────────────────────────────────────────")

equity_results = []

for delta in [None, 0.2, 0.1, 0.0]:
    label = "Unconstrained" if delta is None else f"delta={delta}"

    solver_eq = pywraplp.Solver.CreateSolver("SCIP")
    solver_eq.SetTimeLimit(120_000)  # 2 min per run

    x_eq = [solver_eq.BoolVar(f"x_{j}") for j in J_idx]
    y_eq = {(i, j): solver_eq.BoolVar(f"y_{i}_{j}") for (i, j) in valid_pairs}
    u_eq = [solver_eq.BoolVar(f"u_{i}") for i in I_idx]

    M_eq  = float(MAX_DIST_KM * max(d_A))
    obj_eq = solver_eq.Objective()
    for (i, j) in valid_pairs:
        obj_eq.SetCoefficient(y_eq[i, j], float(d_A[i]) * float(D[i, j]))
    for i in I_idx:
        obj_eq.SetCoefficient(u_eq[i], M_eq * float(d_A[i]))
    obj_eq.SetMinimization()

    # Demand balance
    for i in I_idx:
        if neighbors_of_i[i]:
            solver_eq.Add(
                sum(y_eq[i, j] for j in neighbors_of_i[i]) + u_eq[i] == 1
            )
        else:
            solver_eq.Add(u_eq[i] == 1)

    # Link
    for (i, j) in valid_pairs:
        solver_eq.Add(y_eq[i, j] <= x_eq[j])

    # Budget
    solver_eq.Add(sum(x_eq[j] for j in J_idx) == K)

    # Capacity
    for j in J_idx:
        if neighbors_of_j[j]:
            solver_eq.Add(
                sum(d_A[i] * y_eq[i, j] for i in neighbors_of_j[j]) <= CAP
            )

    # Equity constraint (skip if unconstrained)
    if delta is not None:
        H_pairs    = [(i, j) for (i, j) in valid_pairs if i in H_set]
        nonH_pairs = [(i, j) for (i, j) in valid_pairs if i not in H_set]
        n_H        = len(H_idx)
        n_nonH     = len(result) - n_H

        # avg_dist_H = sum(D[i,j] * y[i,j] for i in H) / n_H
        # avg_dist_nonH = sum(D[i,j] * y[i,j] for i not in H) / n_nonH
        # Constraint: avg_dist_H <= avg_dist_nonH + delta
        # Rearranged: sum_H(D*y)/n_H - sum_nonH(D*y)/n_nonH <= delta
        # Multiply through by n_H * n_nonH:
        # n_nonH * sum_H(D*y) - n_H * sum_nonH(D*y) <= delta * n_H * n_nonH

        solver_eq.Add(
            n_nonH * sum(D[i, j] * y_eq[i, j] for (i, j) in H_pairs) -
            n_H    * sum(D[i, j] * y_eq[i, j] for (i, j) in nonH_pairs)
            <= delta * n_H * n_nonH
        )

    t0_eq    = time.time()
    status_eq = solver_eq.Solve()
    solve_eq  = time.time() - t0_eq

    if status_eq in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE]:
        open_eq      = [j for j in J_idx if x_eq[j].solution_value() > 0.5]
        unserved_eq  = [i for i in I_idx if u_eq[i].solution_value() > 0.5]
        travel_eq    = sum(
            d_A[i] * D[i, j] * y_eq[i, j].solution_value()
            for (i, j) in valid_pairs
        )

        # Compute equity metrics for this solution
        assign_eq = {}
        for (i, j) in valid_pairs:
            if y_eq[i, j].solution_value() > 0.5:
                assign_eq[i] = j

        dist_H_eq    = np.mean([D[i, assign_eq[i]] for i in H_idx
                                 if i in assign_eq]) if H_idx else np.nan
        dist_nonH_eq = np.mean([D[i, assign_eq[i]] for i in I_idx
                                 if i not in H_set and i in assign_eq])
        dist_gap_eq  = dist_H_eq - dist_nonH_eq
        served_eq    = len(result) - len(unserved_eq)

        status_str = ('Optimal' if status_eq == pywraplp.Solver.OPTIMAL
                      else 'Feasible')
        print(f"\n  {label} ({status_str}, {solve_eq:.1f}s):")
        print(f"    Travel cost:    {travel_eq:,.2f} EV-vehicle·km")
        print(f"    Served:         {served_eq} / {len(result)}")
        print(f"    Avg dist H:     {dist_H_eq:.3f} km")
        print(f"    Avg dist non-H: {dist_nonH_eq:.3f} km")
        print(f"    Distance gap:   {dist_gap_eq:+.3f} km")

        equity_results.append({
            "delta":       label,
            "travel_cost": round(travel_eq, 2),
            "served":      served_eq,
            "dist_H":      round(dist_H_eq, 3),
            "dist_nonH":   round(dist_nonH_eq, 3),
            "dist_gap":    round(dist_gap_eq, 3),
            "status":      status_str,
            "solve_time":  round(solve_eq, 1)
        })
    else:
        print(f"\n  {label}: No solution (status {status_eq})")

# ================================
# STEP 3: Summary table
# ================================
print(f"\n── Equity Constraint Summary ────────────────────────────────────")
eq_df = pd.DataFrame(equity_results)
print(eq_df.to_string(index=False))
print(f"────────────────────────────────────────────────────────────────")

eq_df.to_csv("Mountain_View_equity_results.csv", index=False)
print("\nSaved to Mountain_View_equity_results.csv")

## 11. Sensitivity Grid
Runs the MILP across four parameter grids: K ∈ {15,20,25,30}, Q ∈ {300,500,750}, r ∈ {2,3,5} km, adoption rate ∈ {5%,10%,20%}. Each run uses a 60-second time limit.

In [ ]:
# ================================
Sensitivity Grid
# ================================
from ortools.linear_solver import pywraplp
import numpy as np
import pandas as pd
import time

def run_milp(d, K, CAP, max_dist, time_limit=60_000):
    """Run MILP with given parameters. Returns result dict."""
    num_I = len(d)
    num_J = len(sites)
    I_idx = range(num_I)
    J_idx = range(num_J)

    neighbors_i = {i: [j for j in J_idx if D[i, j] <= max_dist] for i in I_idx}
    neighbors_j = {j: [i for i in I_idx if D[i, j] <= max_dist] for j in J_idx}
    valid_pairs = [(i, j) for i in I_idx for j in neighbors_i[i]]
    reachable   = [i for i in I_idx if neighbors_i[i]]

    solver = pywraplp.Solver.CreateSolver("SCIP")
    solver.SetTimeLimit(time_limit)

    x = [solver.BoolVar(f"x_{j}") for j in J_idx]
    y = {(i, j): solver.BoolVar(f"y_{i}_{j}") for (i, j) in valid_pairs}
    u = [solver.BoolVar(f"u_{i}") for i in I_idx]

    M = float(max_dist * max(d)) if len(d) > 0 else 1.0
    obj = solver.Objective()
    for (i, j) in valid_pairs:
        obj.SetCoefficient(y[i, j], float(d[i]) * float(D[i, j]))
    for i in I_idx:
        obj.SetCoefficient(u[i], M * float(d[i]))
    obj.SetMinimization()

    for i in I_idx:
        if neighbors_i[i]:
            solver.Add(sum(y[i, j] for j in neighbors_i[i]) + u[i] == 1)
        else:
            solver.Add(u[i] == 1)

    for (i, j) in valid_pairs:
        solver.Add(y[i, j] <= x[j])

    solver.Add(sum(x[j] for j in J_idx) == K)

    for j in J_idx:
        if neighbors_j[j]:
            solver.Add(sum(d[i] * y[i, j] for i in neighbors_j[j]) <= CAP)

    t0     = time.time()
    status = solver.Solve()
    solve_t = time.time() - t0

    if status in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE]:
        open_sites    = [j for j in J_idx if x[j].solution_value() > 0.5]
        unserved_idx  = [i for i in I_idx if u[i].solution_value() > 0.5]
        served        = num_I - len(unserved_idx)
        travel        = sum(
            d[i] * D[i, j] * y[i, j].solution_value()
            for (i, j) in valid_pairs
        )
        served_demand = sum(d[i] for i in I_idx) - sum(d[i] for i in unserved_idx)
        avg_dist      = travel / served_demand if served_demand > 0 else 0
        avg_load      = served_demand / K if K > 0 else 0
        status_str    = "Optimal" if status == pywraplp.Solver.OPTIMAL else "Feasible"

        return {
            "served":        served,
            "unserved":      len(unserved_idx),
            "served_pct":    round(served / num_I * 100, 1),
            "travel_cost":   round(travel, 2),
            "avg_dist_km":   round(avg_dist, 3),
            "avg_load":      round(avg_load, 1),
            "load_pct":      round(avg_load / CAP * 100, 1),
            "valid_pairs":   len(valid_pairs),
            "status":        status_str,
            "solve_time":    round(solve_t, 1)
        }
    else:
        return {
            "served": None, "unserved": None, "served_pct": None,
            "travel_cost": None, "avg_dist_km": None,
            "avg_load": None, "load_pct": None,
            "valid_pairs": len(valid_pairs),
            "status": "Infeasible", "solve_time": round(solve_t, 1)
        }

# Base demand vector (Scenario A)
d_base = result["demand_A"].values
vehicles_base = result["vehicles"].values

# ================================
# Grid 1: Vary K (station budget)
# ================================
print("── Grid 1: Varying K (stations) ────────────────────────────────")
print(f"  Fixed: Q=500, r=3.0 km, adoption=10%")
rows_K = []
for K_val in [15, 20, 25, 30]:
    r = run_milp(d_base, K_val, 500, 3.0)
    r["K"] = K_val
    rows_K.append(r)
    print(f"  K={K_val}: served={r['served']}/79 ({r['served_pct']}%)  "
          f"travel={r['travel_cost']}  avg_dist={r['avg_dist_km']} km  "
          f"status={r['status']}  time={r['solve_time']}s")

# ================================
# Grid 2: Vary Q (capacity)
# ================================
print("\n── Grid 2: Varying Q (capacity) ────────────────────────────────")
print(f"  Fixed: K=20, r=3.0 km, adoption=10%")
rows_Q = []
for Q_val in [300, 500, 750]:
    r = run_milp(d_base, 20, Q_val, 3.0)
    r["Q"] = Q_val
    rows_Q.append(r)
    print(f"  Q={Q_val}: served={r['served']}/79 ({r['served_pct']}%)  "
          f"travel={r['travel_cost']}  avg_dist={r['avg_dist_km']} km  "
          f"load={r['load_pct']}%  status={r['status']}  time={r['solve_time']}s")

# ================================
# Grid 3: Vary r (proximity radius)
# ================================
print("\n── Grid 3: Varying r (proximity radius) ────────────────────────")
print(f"  Fixed: K=20, Q=500, adoption=10%")
rows_r = []
for r_val in [2.0, 3.0, 5.0]:
    r = run_milp(d_base, 20, 500, r_val)
    r["r_km"] = r_val
    rows_r.append(r)
    print(f"  r={r_val} km: served={r['served']}/79 ({r['served_pct']}%)  "
          f"pairs={r['valid_pairs']}  travel={r['travel_cost']}  "
          f"avg_dist={r['avg_dist_km']} km  status={r['status']}  "
          f"time={r['solve_time']}s")

# ================================
# Grid 4: Vary EV adoption rate
# ================================
print("\n── Grid 4: Varying EV adoption rate ────────────────────────────")
print(f"  Fixed: K=20, Q=500, r=3.0 km")
rows_adopt = []
for rate in [0.05, 0.10, 0.20]:
    d_rate = vehicles_base * rate
    r = run_milp(d_rate, 20, 500, 3.0)
    r["adoption_pct"] = f"{int(rate*100)}%"
    r["total_demand"] = round(sum(d_rate), 1)
    rows_adopt.append(r)
    print(f"  adoption={int(rate*100)}%: demand={r['total_demand']}  "
          f"served={r['served']}/79 ({r['served_pct']}%)  "
          f"travel={r['travel_cost']}  avg_dist={r['avg_dist_km']} km  "
          f"status={r['status']}  time={r['solve_time']}s")

# ================================
# Summary tables
# ================================
print(f"\n── Sensitivity Summary ──────────────────────────────────────────")

df_K = pd.DataFrame(rows_K)[["K","served","served_pct","travel_cost",
                               "avg_dist_km","avg_load","load_pct","status"]]
df_Q = pd.DataFrame(rows_Q)[["Q","served","served_pct","travel_cost",
                               "avg_dist_km","avg_load","load_pct","status"]]
df_r = pd.DataFrame(rows_r)[["r_km","served","served_pct","travel_cost",
                               "avg_dist_km","valid_pairs","status"]]
df_adopt = pd.DataFrame(rows_adopt)[["adoption_pct","total_demand","served",
                                      "served_pct","travel_cost",
                                      "avg_dist_km","status"]]

print("\nK sensitivity:")
print(df_K.to_string(index=False))
print("\nQ sensitivity:")
print(df_Q.to_string(index=False))
print("\nr sensitivity:")
print(df_r.to_string(index=False))
print("\nAdoption rate sensitivity:")
print(df_adopt.to_string(index=False))

# ================================
# Save
# ================================
df_K.to_csv("sensitivity_K.csv", index=False)
df_Q.to_csv("sensitivity_Q.csv", index=False)
df_r.to_csv("sensitivity_r.csv", index=False)
df_adopt.to_csv("sensitivity_adoption.csv", index=False)
print("\nSaved all sensitivity CSVs.")

## 12. Stochastic Demand Analysis — Six Scenarios
Defines six demand scenarios (baseline, high adoption, tenure-weighted, apartment-heavy, equity-growth, commute-heavy). For each scenario, solves for the scenario-optimal solution and evaluates the Scenario A solution via regret analysis. Reports CVaR₀.₉₀ and site selection stability.

In [ ]:
# ================================
Scenario-based stochastic demand model
# ================================
from ortools.linear_solver import pywraplp
import numpy as np
import pandas as pd
import time

# ================================
# STEP 1: Define 6 demand scenarios
# ================================
vehicles   = result["vehicles"].values
renter_share = result["renter_share"].values
d_A        = result["demand_A"].values
d_B        = result["demand_B"].values
hcvi_norm  = result_hcvi["HCVI_norm"].values

# Employment center proxy: distance to downtown Mountain View (Castro St)
DOWNTOWN_LAT = 37.3861
DOWNTOWN_LON = -122.0839
def haversine_scalar(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

dist_to_downtown = np.array([
    haversine_scalar(row["lat"], row["lon"], DOWNTOWN_LAT, DOWNTOWN_LON)
    for _, row in result.iterrows()
])
# Commute proximity weight: closer to downtown = higher public charging need
commute_weight = 1 / (dist_to_downtown + 0.5)
commute_weight = commute_weight / commute_weight.mean()  # normalize to mean 1

# Define scenarios
scenarios = {
    "S1_baseline": {
        "demand": vehicles * 0.10,
        "label": "Baseline (10% uniform stock)",
        "description": "Uniform 10% EV adoption rate applied to all vehicles"
    },
    "S2_high_adoption": {
        "demand": np.minimum(vehicles * 0.20, 500),  # cap at station capacity
        "label": "High adoption (20% stock)",
        "description": "20% EV adoption rate; approaches network capacity limits"
    },
    "S3_tenure_weighted": {
        "demand": d_B,
        "label": "Tenure-weighted need (Scenario B)",
        "description": "Renters 30%, owners 5%; realistic public-charging need"
    },
    "S4_apartment_heavy": {
        "demand": (vehicles * (1 - renter_share) * 0.03 +
                   vehicles * renter_share * 0.50),
        "label": "Apartment-heavy (renters 50%, owners 3%)",
        "description": "Extreme renter-dependency; stress test for dense housing"
    },
    "S5_equity_growth": {
        "demand": vehicles * 0.10 * (1 + hcvi_norm),
        "label": "Equity-growth (HCVI-weighted)",
        "description": "Latent demand growth proportional to vulnerability index"
    },
    "S6_commute_heavy": {
        "demand": vehicles * 0.10 * commute_weight * renter_share,
        "label": "Commute-heavy (downtown proximity × renter)",
        "description": "Public charging need weighted by commute pattern proxy"
    },
}

print(f"── Demand Scenario Summary ──────────────────────────────────────")
print(f"  {'Scenario':<30} {'Total demand':>14} {'Max block group':>16}")
print(f"  {'-'*60}")
for s_id, s in scenarios.items():
    d = s["demand"]
    print(f"  {s['label']:<30} {d.sum():>14.1f} {d.max():>16.1f}")
print(f"────────────────────────────────────────────────────────────────")

# ================================
# STEP 2: Helper — solve MILP for a given demand vector
# ================================
def solve_scenario(d_vec, K=20, CAP=500, max_dist=3.0, time_limit=60_000,
                   label=""):
    num_I = len(d_vec)
    num_J = len(sites)
    I_idx = range(num_I)
    J_idx = range(num_J)

    # Cap demand at CAP to avoid infeasibility from single large demand point
    d_vec = np.minimum(d_vec, CAP * 0.95)

    neighbors_i = {i: [j for j in J_idx if D[i,j] <= max_dist] for i in I_idx}
    neighbors_j = {j: [i for i in I_idx if D[i,j] <= max_dist] for j in J_idx}
    valid_pairs = [(i,j) for i in I_idx for j in neighbors_i[i]]

    solver = pywraplp.Solver.CreateSolver("SCIP")
    solver.SetTimeLimit(time_limit)

    x = [solver.BoolVar(f"x_{j}") for j in J_idx]
    y = {(i,j): solver.BoolVar(f"y_{i}_{j}") for (i,j) in valid_pairs}
    u = [solver.BoolVar(f"u_{i}") for i in I_idx]

    M = float(max_dist * max(d_vec)) if len(d_vec) > 0 else 1.0
    obj = solver.Objective()
    for (i,j) in valid_pairs:
        obj.SetCoefficient(y[i,j], float(d_vec[i]) * float(D[i,j]))
    for i in I_idx:
        obj.SetCoefficient(u[i], M * float(d_vec[i]))
    obj.SetMinimization()

    for i in I_idx:
        if neighbors_i[i]:
            solver.Add(sum(y[i,j] for j in neighbors_i[i]) + u[i] == 1)
        else:
            solver.Add(u[i] == 1)

    for (i,j) in valid_pairs:
        solver.Add(y[i,j] <= x[j])

    solver.Add(sum(x[j] for j in J_idx) == K)

    for j in J_idx:
        if neighbors_j[j]:
            solver.Add(sum(d_vec[i] * y[i,j] for i in neighbors_j[j]) <= CAP)

    t0 = time.time()
    status = solver.Solve()
    solve_t = time.time() - t0

    if status in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE]:
        open_sites   = [j for j in J_idx if x[j].solution_value() > 0.5]
        unserved     = [i for i in I_idx if u[i].solution_value() > 0.5]
        travel       = sum(
            d_vec[i] * D[i,j] * y[i,j].solution_value()
            for (i,j) in valid_pairs
        )
        served_dem   = sum(d_vec) - sum(d_vec[i] for i in unserved)
        avg_dist     = travel / served_dem if served_dem > 0 else 0

        return {
            "status":      "Optimal" if status == pywraplp.Solver.OPTIMAL else "Feasible",
            "open_sites":  set(open_sites),
            "site_ids":    set(sites["site_id"].iloc[j] for j in open_sites),
            "unserved":    len(unserved),
            "served":      len(I_idx) - len(unserved),
            "travel_cost": travel,
            "served_dem":  served_dem,
            "avg_dist":    avg_dist,
            "solve_time":  solve_t
        }
    else:
        return None

# ================================
# STEP 3: Solve best solution for each scenario
# ================================
print(f"\n── Solving Best Solution for Each Scenario ──────────────────────")
best_solutions = {}
for s_id, s in scenarios.items():
    print(f"  Solving {s['label']}...", end=" ", flush=True)
    result_s = solve_scenario(s["demand"], label=s_id)
    if result_s:
        best_solutions[s_id] = result_s
        print(f"travel={result_s['travel_cost']:,.1f}  "
              f"served={result_s['served']}/79  "
              f"status={result_s['status']}  "
              f"time={result_s['solve_time']:.1f}s")
    else:
        print("FAILED")

# ================================
# STEP 4: Evaluate Scenario A solution under each scenario
# ================================
print(f"\n── Evaluating Scenario A Solution Across All Scenarios ──────────")

# Scenario A open site indices
A_open_ids = set(solution_df["site_id"])
A_open_idx = set(
    sites[sites["site_id"] == sid].index[0]
    for sid in A_open_ids
)

def evaluate_fixed_solution(open_idx, d_vec, CAP=500, max_dist=3.0):
    """Evaluate a fixed set of open stations under a given demand vector."""
    num_I = len(d_vec)
    neighbors_i = {
        i: [j for j in open_idx if D[i,j] <= max_dist]
        for i in range(num_I)
    }
    assignment = {}
    load = {j: 0.0 for j in open_idx}

    for i in sorted(range(num_I), key=lambda i: -d_vec[i]):
        candidates = [
            j for j in neighbors_i[i]
            if load[j] + d_vec[i] <= CAP
        ]
        if candidates:
            j = min(candidates, key=lambda j: D[i,j])
            assignment[i] = j
            load[j] += d_vec[i]

    served    = len(assignment)
    unserved  = num_I - served
    travel    = sum(d_vec[i] * D[i, assignment[i]] for i in assignment)
    served_dem = sum(d_vec[i] for i in assignment)
    avg_dist  = travel / served_dem if served_dem > 0 else 0

    return {
        "served":      served,
        "unserved":    unserved,
        "travel_cost": travel,
        "served_dem":  served_dem,
        "avg_dist":    avg_dist
    }

eval_results = {}
for s_id, s in scenarios.items():
    d_vec = np.minimum(s["demand"], 500 * 0.95)
    ev = evaluate_fixed_solution(A_open_idx, d_vec)
    eval_results[s_id] = ev
    print(f"  {s['label']:<35} travel={ev['travel_cost']:>10,.1f}  "
          f"served={ev['served']}/79  "
          f"avg_dist={ev['avg_dist']:.3f} km")

# ================================
# STEP 5: Regret analysis
# ================================
print(f"\n── Regret Analysis ──────────────────────────────────────────────")
print(f"  Regret = cost of Scenario A solution under scenario s")
print(f"           minus best possible cost under scenario s")
print(f"  {'Scenario':<35} {'Best*':>10} {'ScenA eval':>12} {'Regret':>10} {'Regret%':>8}")
print(f"  {'-'*75}")

regrets = {}
for s_id, s in scenarios.items():
    if s_id in best_solutions and best_solutions[s_id]:
        best_cost = best_solutions[s_id]["travel_cost"]
        eval_cost = eval_results[s_id]["travel_cost"]
        regret    = eval_cost - best_cost
        regret_pct = (regret / best_cost * 100) if best_cost > 0 else 0
        regrets[s_id] = regret
        print(f"  {s['label']:<35} {best_cost:>10,.1f} {eval_cost:>12,.1f} "
              f"{regret:>10,.1f} {regret_pct:>7.1f}%")

max_regret_id  = max(regrets, key=regrets.get)
max_regret_val = regrets[max_regret_id]
print(f"\n  Max regret: {scenarios[max_regret_id]['label']} "
      f"({max_regret_val:,.1f} EV-vehicle·km)")

# ================================
# STEP 6: CVaR_0.9 of travel cost
# ================================
travel_costs = [eval_results[s_id]["travel_cost"] for s_id in scenarios]
travel_costs_arr = np.array(sorted(travel_costs))
alpha    = 0.90
n_scen   = len(travel_costs_arr)
cutoff   = int(np.ceil(alpha * n_scen))
cvar_90  = travel_costs_arr[cutoff-1:].mean()
exp_cost = travel_costs_arr.mean()

print(f"\n── Risk Metrics ─────────────────────────────────────────────────")
print(f"  Scenario travel costs: {[round(c,1) for c in sorted(travel_costs)]}")
print(f"  Expected travel cost (avg across scenarios): {exp_cost:,.2f}")
print(f"  CVaR_0.90 (avg of top 10% worst scenarios):  {cvar_90:,.2f}")
print(f"────────────────────────────────────────────────────────────────")

# ================================
# STEP 7: Site selection stability
# ================================
print(f"\n── Site Selection Stability ─────────────────────────────────────")
print(f"  How often each site appears across the 6 optimal solutions:")

site_freq = {}
for s_id, sol in best_solutions.items():
    for sid in sol["site_ids"]:
        site_freq[sid] = site_freq.get(sid, 0) + 1

# Classify stability
stable_core    = {sid for sid, freq in site_freq.items() if freq >= 5}
moderate       = {sid for sid, freq in site_freq.items() if freq == 3 or freq == 4}
scenario_spec  = {sid for sid, freq in site_freq.items() if freq <= 2}

print(f"  Stable core (≥5/6 scenarios):      {len(stable_core)} sites: {sorted(stable_core)}")
print(f"  Moderate (3-4/6 scenarios):         {len(moderate)} sites")
print(f"  Scenario-specific (≤2/6 scenarios): {len(scenario_spec)} sites")

# Scenario A sites in stable core
A_in_core = A_open_ids & stable_core
print(f"\n  Scenario A sites in stable core:   {len(A_in_core)}/20 → {sorted(A_in_core)}")
print(f"────────────────────────────────────────────────────────────────")

# ================================
# STEP 8: Summary table
# ================================
rows = []
for s_id, s in scenarios.items():
    best = best_solutions.get(s_id, {})
    ev   = eval_results[s_id]
    reg  = regrets.get(s_id, None)
    rows.append({
        "Scenario":          s["label"],
        "Total demand":      round(np.minimum(s["demand"], 475).sum(), 1),
        "Best* travel cost": round(best.get("travel_cost", np.nan), 1),
        "Best* served":      best.get("served", "—"),
        "ScenA travel cost": round(ev["travel_cost"], 1),
        "ScenA served":      ev["served"],
        "Regret":            round(reg, 1) if reg is not None else "—",
        "Best* status":      best.get("status", "—"),
    })

summary_df = pd.DataFrame(rows)
print(f"\n── Full Scenario Summary Table ──────────────────────────────────")
print(summary_df.to_string(index=False))
print(f"────────────────────────────────────────────────────────────────")

summary_df.to_csv("Mountain_View_scenario_analysis.csv", index=False)
print(f"\nSaved to Mountain_View_scenario_analysis.csv")
print(f"\nExpected travel cost: {exp_cost:,.2f} EV-vehicle·km")
print(f"CVaR_0.90:            {cvar_90:,.2f} EV-vehicle·km")
print(f"Max regret scenario:  {scenarios[max_regret_id]['label']}")
print(f"Max regret value:     {max_regret_val:,.2f} EV-vehicle·km")

## 13. Road-Network Distance Validation
Downloads the Mountain View OSM drive network and computes actual road-network shortest paths for all 79 assigned demand-station pairs. Compares against haversine distances to quantify approximation error.

In [ ]:
# ================================
Road-network distance robustness check
# ================================
import osmnx as ox
import networkx as nx
import numpy as np
import pandas as pd

print("Downloading Mountain View road network...")
G = ox.graph_from_place("Mountain View, California, USA", network_type="drive")
print(f"Graph: {len(G.nodes)} nodes, {len(G.edges)} edges")

# ================================
# STEP 1: For each assigned (block group, station) pair in MILP solution,
# compute road-network distance and compare to haversine
# ================================

# Rebuild assignment from solution
open_site_ids     = set(solution_df["site_id"])
open_site_indices = [
    sites[sites["site_id"] == sid].index[0]
    for sid in open_site_ids
]

# Assign each block group to nearest open station (same logic as equity cell)
milp_assignment = {}
for i in range(len(result)):
    candidates = [j for j in open_site_indices if D[i, j] <= MAX_DIST_KM]
    if candidates:
        milp_assignment[i] = min(candidates, key=lambda j: D[i, j])

print(f"\nComputing road-network distances for {len(milp_assignment)} assignments...")

road_results = []
errors = 0

for i, j in milp_assignment.items():
    bg_lat  = result["lat"].iloc[i]
    bg_lon  = result["lon"].iloc[i]
    st_lat  = sites["lat"].iloc[j]
    st_lon  = sites["lon"].iloc[j]
    hav_dist = D[i, j]

    try:
        # Find nearest network nodes
        orig_node = ox.distance.nearest_nodes(G, bg_lon, bg_lat)
        dest_node = ox.distance.nearest_nodes(G, st_lon, st_lat)

        # Shortest path length in meters
        road_m = nx.shortest_path_length(
            G, orig_node, dest_node, weight="length"
        )
        road_km = road_m / 1000.0
        ratio   = road_km / hav_dist if hav_dist > 0 else np.nan

        road_results.append({
            "block_group":  result["block_group"].iloc[i],
            "site_id":      sites["site_id"].iloc[j],
            "haversine_km": round(hav_dist, 4),
            "road_km":      round(road_km, 4),
            "ratio":        round(ratio, 4),
            "diff_km":      round(road_km - hav_dist, 4)
        })

    except Exception:
        errors += 1

print(f"  Computed: {len(road_results)} pairs")
print(f"  Errors (unreachable nodes): {errors}")

# ================================
# STEP 2: Summary statistics
# ================================
df_road = pd.DataFrame(road_results)

print(f"\n── Road-Network vs Haversine Distance Comparison ───────────────")
print(f"  {'Metric':<40} {'Haversine':>12} {'Road network':>14}")
print(f"  {'-'*66}")
print(f"  {'Mean distance (km)':<40} {df_road['haversine_km'].mean():>12.3f} {df_road['road_km'].mean():>14.3f}")
print(f"  {'Median distance (km)':<40} {df_road['haversine_km'].median():>12.3f} {df_road['road_km'].median():>14.3f}")
print(f"  {'Max distance (km)':<40} {df_road['haversine_km'].max():>12.3f} {df_road['road_km'].max():>14.3f}")
print(f"  {'Mean road/haversine ratio':<40} {df_road['ratio'].mean():>12.3f} {'':>14}")
print(f"  {'Median road/haversine ratio':<40} {df_road['ratio'].median():>12.3f} {'':>14}")
print(f"  {'Mean absolute difference (km)':<40} {df_road['diff_km'].mean():>12.3f} {'':>14}")
print(f"  {'-'*66}")

# D95 comparison
d95_hav  = df_road["haversine_km"].quantile(0.95)
d95_road = df_road["road_km"].quantile(0.95)
print(f"  {'D95 (km)':<40} {d95_hav:>12.3f} {d95_road:>14.3f}")
print(f"────────────────────────────────────────────────────────────────")

# Worst-case pairs
print(f"\nTop 5 pairs by road/haversine ratio:")
print(df_road.nlargest(5, "ratio")[
    ["block_group", "site_id", "haversine_km", "road_km", "ratio"]
].to_string(index=False))

# ================================
# STEP 3: Save
# ================================
df_road.to_csv("Mountain_View_road_vs_haversine.csv", index=False)
print(f"\nSaved to Mountain_View_road_vs_haversine.csv")

## 14. Interactive Solution Maps
### MILP Solution Map
Blue circles: selected charging stations. Green circles: demand points sized by EV demand.

In [ ]:
!pip install folium -q
import folium

m = folium.Map(location=[37.397, -122.078], zoom_start=13, tiles="OpenStreetMap")

for _, row in result.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=max(4, min(10, row["demand_A"] / 12)),
        color="#1D9E75", fill=True, fill_color="#5DCAA5",
        fill_opacity=0.7, weight=1.5,
        popup=folium.Popup(
            f"<b>Block group:</b> {row['block_group']}<br>"
            f"<b>Population:</b> {int(row['population'])}<br>"
            f"<b>EV demand (A):</b> {row['demand_A']:.1f}<br>"
            f"<b>EV demand (B):</b> {row['demand_B']:.1f}<br>"
            f"<b>Renter share:</b> {row['renter_share']:.2f}",
            max_width=220)
    ).add_to(m)

for _, row in solution_df.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=12, color="#185FA5", fill=True,
        fill_color="#378ADD", fill_opacity=0.9, weight=2.5,
        popup=folium.Popup(
            f"<b>Station {row['site_id']}</b><br>"
            f"<b>Assigned:</b> {row['n_assigned']}<br>"
            f"<b>Demand:</b> {row['stn_demand']}<br>"
            f"<b>Avg dist:</b> {row['avg_dist_km']} km<br>"
            f"<b>Load:</b> {row['load_pct']}%",
            max_width=200)
    ).add_to(m)
    folium.Marker(
        location=[row["lat"], row["lon"]],
        icon=folium.DivIcon(
            html=f'<div style="font-size:9px;font-weight:bold;color:#042C53;'
                 f'white-space:nowrap;margin-top:-6px;margin-left:14px;">'
                 f'{row["site_id"]}</div>',
            icon_size=(40, 12), icon_anchor=(0, 6))
    ).add_to(m)

legend_html = """
<div style="position:fixed;bottom:30px;right:30px;z-index:1000;
     background:white;border:1px solid #ccc;border-radius:8px;
     padding:12px 16px;font-size:13px;font-family:sans-serif;line-height:2;">
  <b>Mountain View EV Charging — CFLP Solution</b><br>
  <span style="display:inline-block;width:12px;height:12px;border-radius:50%;
    background:#378ADD;margin-right:6px;vertical-align:middle;"></span>
    Charging station (20 built)<br>
  <span style="display:inline-block;width:12px;height:12px;border-radius:50%;
    background:#5DCAA5;margin-right:6px;vertical-align:middle;"></span>
    Demand point (block group)<br>
  <span style="font-size:11px;color:#888;">Click any point for details</span>
</div>"""
m.get_root().html.add_child(folium.Element(legend_html))
m.save("Mountain_View_CFLP_map.html")
print("Saved.")

print("Map saved to Mountain_View_CFLP_map.html")


### Heuristic Solution Map
Blue circles: active stations. Green circles: demand points sized by EV demand.

In [ ]:
import folium

m2 = folium.Map(location=[37.397, -122.078], zoom_start=13, tiles="OpenStreetMap")

for _, row in result.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=max(4, min(10, row["demand_A"] / 12)),
        color="#1D9E75", fill=True, fill_color="#5DCAA5",
        fill_opacity=0.7, weight=1.5,
        popup=folium.Popup(
            f"<b>Block group:</b> {row['block_group']}<br>"
            f"<b>EV demand (A):</b> {row['demand_A']:.1f}",
            max_width=200)
    ).add_to(m2)

for _, row in heuristic_df.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=12, color="#185FA5", fill=True,
        fill_color="#378ADD", fill_opacity=0.9, weight=2.5,
        popup=folium.Popup(
            f"<b>Station {row['site_id']}</b><br>"
            f"<b>Assigned:</b> {row['n_assigned']}<br>"
            f"<b>Demand:</b> {row['stn_demand']}<br>"
            f"<b>Avg dist:</b> {row['avg_dist_km']} km<br>"
            f"<b>Load:</b> {row['load_pct']}%",
            max_width=200)
    ).add_to(m2)
    folium.Marker(
        location=[row["lat"], row["lon"]],
        icon=folium.DivIcon(
            html=f'<div style="font-size:9px;font-weight:bold;color:#042C53;'
                 f'white-space:nowrap;margin-top:-6px;margin-left:14px;">'
                 f'{row["site_id"]}</div>',
            icon_size=(40, 12), icon_anchor=(0, 6))
    ).add_to(m2)

legend_html2 = """
<div style="position:fixed;bottom:30px;right:30px;z-index:1000;
     background:white;border:1px solid #ccc;border-radius:8px;
     padding:12px 16px;font-size:13px;font-family:sans-serif;line-height:2;">
  <b>Mountain View EV Charging — Heuristic Solution</b><br>
  <span style="display:inline-block;width:12px;height:12px;border-radius:50%;
    background:#378ADD;margin-right:6px;vertical-align:middle;"></span>
    Charging station (active)<br>
  <span style="display:inline-block;width:12px;height:12px;border-radius:50%;
    background:#5DCAA5;margin-right:6px;vertical-align:middle;"></span>
    Demand point (block group)<br>
  <span style="font-size:11px;color:#888;">Click any point for details</span>
</div>"""
m2.get_root().html.add_child(folium.Element(legend_html2))
m2.save("Mountain_View_heuristic_map.html")
print("Saved.")

print("Map saved to Mountain_View_heuristic_map.html")
